## Water Injection Dredging 
This notebook is made to simulate the workflow of a Water Injection Dredger (WID). This method is conducted by Water Injection Dredgers (WID) by injecting water with a jetbar and multiple nozzles to the sediment bed, fluidize the sediments, and leads the sediments to the open sea using currents and waves.

The WID module is generated by defining a "processor_wid" mixin and "produce_amount_activity" activity.

The production via "produce_amount_activity" continues until the condition of WhileActivity meets. There will be no discharging location in this case.

#### 0. Import libraries

In [11]:
# import datetime, time
import simpy
import shapely.geometry
import pandas as pd
import inspect
import sys 
import os 
import matplotlib.pyplot as plt

import openclsim.core as core
import openclsim.model as model
import openclsim.plot as plot
import openclsim.plugins as plugins

In [12]:
simulation_start = 0
my_env = simpy.Environment(initial_time=simulation_start)

In [13]:
Site = type(
    "Site",
    (
        core.Identifiable,
        core.Log,
        core.Locatable,
        core.HasContainer,
        core.HasResource,
        core.HasSoilWID,
    ),
    {},
)
TransportProcessingResource = type(
    "TransportProcessingResource",
    (   
        core.HasJetBeam,
        core.HasJetPipe,
        core.HasPropellerWID,
        core.HasSoilWID,
        core.ContainerDependentMovable, 
        core.HasResource,
        core.Processor,
        core.Identifiable,
        core.Log,
    ),
    {},
)

In [14]:
location_port = shapely.geometry.Point(1.41941, 51.32966)           # location of the Port of Ramsgate
location_dredging = shapely.geometry.Point(1.420900, 51.325379)     # location of the dredging site

data_dredging = {
    "env": my_env,
    "name": "dredging_site",
    "geometry": location_dredging,    
    "capacity":150000,
    "level":150000,
    "rho_particle": 2600,
    "rho_water": 1025,
    "rho_cloud": 1070,
    "rho_situ": 1200,
    "saturation": 100,
    "dredged_volume": 135000,
    "dredged_area": 108367,
    "grain_size_diameter": 175,
    "mass_flux_coefficient": 0.1,
    "initial_porosity": 0.44,
}

data_port = {
    "env": my_env,
    "name": "port_site",
    "geometry": location_port,    
    "capacity":150000,
    "level":0,
    "rho_particle": 2600,
    "rho_water": 1025,
    "rho_cloud": 1070,
    "rho_situ": 1200,
    "saturation": 100,
    "dredged_volume": 135000,
    "dredged_area": 108367,
    "grain_size_diameter": 175,
    "mass_flux_coefficient": 0.1,
    "initial_porosity": 0.44,
}

# instantiate from_site
dredging_site = Site(**data_dredging)
port_site = Site(**data_port)

##### 3.2. Create vessel object(s)

In [15]:
# prepare input data for vessel_01
data_vessel01 = {
    "env": my_env,
    "name": "vessel01",
    "geometry": location_port,  
    "capacity":1740,        # capacity doesn't matter in the WID as no re-allocation is done
    "compute_v": lambda x: 10,
    "rho_particle": 2600,
    "rho_water": 1025,
    "rho_cloud": 1070,
    "rho_situ": 1200,
    "saturation": 100,
    "dredged_volume": 135000,
    "dredged_area": 108367,
    "grain_size_diameter": 175,
    "mass_flux_coefficient": 0.1,
    "initial_porosity": 0.44,
    "jet_beam_width": 12,
    "jet_beam_diameter": 0.811,
    "n_nozzles": 41,
    "velocity_water_jets": 5,
    "nozzle_diameter": 0.076,
    "water_jet_production": 1,
    "stand_of_distance": 0.4,
    "dredging_speed": 0.51,
    "nozzle_pressure": 123,
    "undrained_shear_strength": 0.5,
    "jet_contraction_coefficient": 0.75,
    "attachment_cable_length": 22.72,
    "jet_pipe_diameter": 0.811,
    "rhu_jet_pipe": 7085,
    "c_drag": 0.9,
    "jet_pipe_length": 30,

}
# instantiate vessel_01 
vessel01 = TransportProcessingResource(**data_vessel01)

TypeError: __init__() missing 10 required positional arguments: 'rho_situ', 'rho_particle', 'rho_water', 'rho_cloud', 'saturation', 'dredged_volume', 'dredged_area', 'mass_flux_coefficient', 'grain_size_diameter', and 'initial_porosity'

##### 3.3. Create activity/activities

In [ ]:
# initialise registry
registry = {}

In [ ]:
ProduceActivity = type(
    "ProduceActivity",
    (
        core.HasWIDProduction,
        model.ShiftAmountActivity,  # the order is critical!
    ),
    {},
)

sub_processes = [
    model.MoveActivity(
        env=my_env,
        name="dredging_trip",
        registry=registry,
        mover = vessel01,
        destination = dredging_site
    ),
    ProduceActivity(
        env=my_env,
        name="dredging",
        registry=registry,
        processor = vessel01,
        origin = dredging_site,
        destination = vessel01,
        amount = 1740,
        duration = 100,
        jet_beam_width = 12,
        jet_beam_diameter = 0.811,
        n_nozzles = 41,
        velocity_water_jets = 5,
        nozzle_diameter = 41,
        water_jet_production = 3, 
        stand_of_distance = 0.4,
        dredging_speed = 0.5144,
        nozzle_pressure = 123,
        undrained_shear_strength = 25,
        jet_contraction_coefficient = 0.75,
        jet_pipe_length = 30,
        attachment_cable_length = 22.72,
        jet_pipe_diameter = 0.811,
        rhu_jet_pipe = 7085,
        c_drag = 0.9,
        rho_situ = 1200,
        rho_particle = 2730,
        rho_water = 1025,
        rho_cloud = 1070,
        saturation = 100,
        dredged_volume = 135000,
        dredged_area = 108367,
        mass_flux_coefficient = 0.1,
        grain_size_diameter = 175,
        initial_porosity = 0.34
    ),
    model.MoveActivity(
        env=my_env,
        name="port_trip",
        registry=registry,
        mover = vessel01,
        destination = port_site
    ),     
]

sequential_activity = model.SequentialActivity(
    env=my_env,
    registry=registry,
    name="Sequence",
    sub_processes = sub_processes
)

# create a while activity that executes the 'sequential activity' while the stop condition is not triggered
while_activity = model.WhileActivity(
    env=my_env,
    name='While activity',
    registry=registry,
    sub_processes=[sequential_activity],
    condition_event=[
        {"type":"container", "concept":dredging_site,"state":"empty"}
    ],
)

TypeError: __init__() got multiple values for argument 'jet_beam_width'

#### 4. Register processes and run simpy

In [ ]:
model.register_processes(while_activity)
my_env.run()

AttributeError: type object 'HasJetPipe' has no attribute 'jet_pipe_diameter'

In [ ]:
display(plot.get_log_dataframe(vessel01, [*sub_processes]))

,Activity,Timestamp,ActivityState,container level,geometry
0,dredging_trip,1970-01-01 00:00:00.000000,START,0.0,POINT (1.41941 51.32966)
1,dredging_trip,1970-01-01 00:00:48.747252,STOP,0.0,POINT (1.4209 51.325379)
2,dredging,1970-01-01 00:00:48.747252,START,0.0,POINT (1.4209 51.325379)
3,dredging,1970-01-01 00:02:28.747252,STOP,0.0,POINT (1.4209 51.325379)
4,port_trip,1970-01-01 00:02:28.747252,START,0.0,POINT (1.4209 51.325379)
...,...,...,...,...,...
517,dredging_trip,1970-01-01 04:43:53.274588,STOP,0.0,POINT (1.4209 51.325379)
518,dredging,1970-01-01 04:43:53.274588,START,0.0,POINT (1.4209 51.325379)
519,dredging,1970-01-01 04:45:33.274588,STOP,0.0,POINT (1.4209 51.325379)
520,port_trip,1970-01-01 04:45:33.274588,START,0.0,POINT (1.4209 51.325379)
